# 03. 모델 평가 — Batch Transform

## 이 노트북에서 배우는 것
- 학습된 모델을 **Batch Transform**(대량 오프라인 추론)으로 테스트셋에 적용
- 다중 클래스 확률 출력(softprob)을 **예측 라벨**로 변환 (argmax)
- **혼동 행렬 / 클래스별 precision·recall·F1 / macro-F1** 로 성능 해석
- 왜 정확도(accuracy)만 보면 안 되는지 (클래스 불균형)

`02_training.ipynb` 를 먼저 실행했어야 합니다.

## 0. 환경 복원

학습 작업 이름으로 Estimator를 다시 붙여(attach) 학습된 모델을 재사용합니다.

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.estimator import Estimator

sess = sagemaker.Session()
role = get_execution_role()
region = sess.boto_region_name

%store -r bucket prefix training_job_name test_x_s3 test_y_s3

# 이미 완료된 학습 작업에 다시 연결 -> 학습된 모델을 그대로 사용
xgb = Estimator.attach(training_job_name, sagemaker_session=sess)
print("attached:", training_job_name)

## 1. Batch Transform 실행

`Transformer` 는 엔드포인트를 상시 띄우지 않고, 대량의 입력을 한 번에 채점한 뒤 리소스를 정리합니다. 테스트 **피처(라벨 없음)** 파일을 입력으로 줍니다.

In [ ]:
batch_output = f"s3://{bucket}/{prefix}/batch-eval"

transformer = xgb.transformer(
    instance_count=1,
    # TODO: 배치 변환 인스턴스 타입을 지정하세요 (예: "ml.m5.xlarge").
    instance_type=____,
    output_path=batch_output,
    assemble_with="Line",
    accept="text/csv",
)

# TODO: 입력은 CSV이고 한 줄이 한 샘플입니다. content_type 과 split_type 을 지정하세요.
#  - content_type="text/csv", split_type="Line"
#  - 참고: https://sagemaker.readthedocs.io/en/v2.219.0/api/inference/transformer.html
transformer.transform(test_x_s3, content_type=____, split_type=____)
transformer.wait()
print("batch output:", transformer.output_path)

## 2. 예측 결과 내려받기 & 라벨 변환

Batch Transform 출력은 입력 파일명에 `.out` 이 붙습니다. 각 줄은 3개 클래스 확률입니다.

In [ ]:
import json
import numpy as np
import pandas as pd
from sagemaker.s3 import S3Downloader

S3Downloader.download(transformer.output_path, "batch_out")
S3Downloader.download(test_y_s3, "batch_out")

proba = np.array([json.loads(line) for line in open("batch_out/test_x.csv.out")])
y_true = pd.read_csv("batch_out/test_y.csv", header=None).values.ravel()

# TODO: 각 행에서 확률이 가장 큰 클래스의 인덱스를 예측 라벨로 만드세요.
#  - 힌트: numpy 의 argmax 를 axis=1 로 사용
y_pred = ____

print("samples:", len(y_true), "| pred shape:", proba.shape)
pd.DataFrame(proba, columns=["p_Dropout", "p_Enrolled", "p_Graduate"]).head()

## 3. 성능 지표

혼동 행렬과 클래스별 리포트를 확인합니다.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

labels = ["Dropout", "Enrolled", "Graduate"]

print("Confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=labels, columns=labels))
print()
print(classification_report(y_true, y_pred, target_names=labels, digits=3))

acc = accuracy_score(y_true, y_pred)
# TODO: 클래스 불균형에서는 macro 평균 F1이 더 공정한 지표입니다. average 인자를 지정하세요.
macro_f1 = f1_score(y_true, y_pred, average=____)
print(f"accuracy : {acc:.3f}")
print(f"macro F1 : {macro_f1:.3f}")

### 해석 포인트
- `Enrolled`(소수 클래스)의 **recall**이 다른 클래스보다 낮게 나오는지 확인하세요. 전체 정확도가
  높아도 소수 클래스를 잘 못 맞히는 경우가 흔합니다.
- 그래서 **macro-F1**(클래스별 F1의 단순 평균)을 함께 봅니다. 다음 노트북(`04`)에서
  하이퍼파라미터 튜닝으로 이 지표를 끌어올려 봅니다.

✅ **완료!** 다음은 `04_tuning.ipynb`.